In [0]:
df_laptimes = spark.read.csv("/Volumes/gr5069/raw/f1_data/lap_times.csv",header= True)

In [0]:
df_pitstops= spark.read.csv("/Volumes/gr5069/raw/f1_data/pit_stops.csv",header= True)

In [0]:
df_pitstops.head(5)

Question 1: What was the average time each driver spent at the pit stop for each race? Provide also the slowest and fastest pit stop in each race.

Logic: To answer this question, I grouped the pit stop data by raceId and driverId since each driver can have multiple pit stops within a race. I then calculated the average pit stop time, as well as the fastest and slowest pit stops using the milliseconds column. This column was used because it contains numeric values that allow accurate aggregation.

In [0]:
from pyspark.sql.functions import avg, min, max, col

df_pitstops_clean = df_pitstops.withColumn(
    "milliseconds", col("milliseconds").cast("int")
)

pitstop_stats = df_pitstops_clean.groupBy("raceId", "driverId").agg(
    avg("milliseconds").alias("avg_pit_time"),
    min("milliseconds").alias("fastest_pit"),
    max("milliseconds").alias("slowest_pit")
)

pitstop_stats.orderBy("raceId", "driverId").show()


Explanation: First, I converted the milliseconds column to an integer to ensure numerical calculations work properly. Then I used groupBy("raceId", "driverId") to group pit stops by driver and race. After grouping, I applied aggregation functions: avg() to compute the average pit stop time, min() to find the fastest stop, and max() to find the slowest stop. Finally, I displayed the results to show the pit stop statistics for each driver in each race.

Question 2: Rank order by finishing position the average time spent at the pit stop in each race.

Logic: To rank drivers by finishing position and compute average pit stop time, I joined the pit_stops dataset with the results dataset using raceId and driverId. The results dataset contains the finishing position for each driver. After joining, I grouped the data by race and finishing position, then calculated the average pit stop time using the milliseconds column. Finally, I ordered the results by finishing position within each race.

In [0]:
df_results = spark.read.csv(
    "/Volumes/gr5069/raw/f1_data/results.csv",
    header=True
)


In [0]:
from pyspark.sql.functions import avg, col

df_pitstops_clean = df_pitstops.withColumn(
    "milliseconds", col("milliseconds").cast("int")
)

df_join = df_results.join(
    df_pitstops_clean,
    ["raceId", "driverId"],
    "inner"
)

ranked_pit = df_join.groupBy(
    "raceId",
    "position",
    "driverId"
).agg(
    avg(df_pitstops_clean["milliseconds"]).alias("avg_pit_time")
)

ranked_pit.orderBy("raceId", "position").show()


Explanation: First, I converted the milliseconds column to integer to allow aggregation. Then I joined the pit stop dataset with the results dataset using raceId and driverId to obtain finishing positions. After joining, I grouped the data by race and position, and calculated the average pit stop time using avg(milliseconds). Finally, I sorted the output by race and finishing position to rank drivers based on race results.

Question 3: Insert the missing code (e.g: ALO for Alonso) for drivers based on the 'drivers' dataset. Explain your logic.

Logic: To fill in the missing driver codes, I used the drivers dataset and checked which rows had a missing value in the code column. Since the example given is ALO for Alonso, I used the driver’s surname to create a replacement code by taking the first three letters of the surname and converting them to uppercase. This gives a consistent three letter abbreviation for drivers whose code is missing, while keeping the original code for drivers where it already exists.

In [0]:
df_drivers = spark.read.csv(
    "/Volumes/gr5069/raw/f1_data/drivers.csv",
    header=True
)


In [0]:
from pyspark.sql.functions import col, when, upper, substring

df_drivers_filled = df_drivers.withColumn(
    "code",
    when(
        col("code").isNull() | (col("code") == ""),
        upper(substring(col("surname"), 1, 3))
    ).otherwise(col("code"))
)

df_drivers_filled.select("driverId", "forename", "surname", "code").show()


Explanation: First, I used withColumn() to update the code column. Then I used when() to check whether the existing code was missing, either as NULL or an empty string. If it was missing, I created a new code by applying substring() to take the first three letters of the surname and upper() to convert them to uppercase. If a driver already had a code, otherwise() kept the original value unchanged. This approach preserves existing data and only fills in the missing codes.

Question 4: Who is the youngest and the oldest driver in each race? Create a new column called “Age”. Explain your definition of "age".

Logic: To find the youngest and oldest driver in each race, I calculated each driver’s age using their date of birth from the drivers dataset and the race date from the races dataset. I joined the drivers, results, and races tables using driverId and raceId. Age is defined as the driver’s age in years on the day of the race. After creating the Age column, I grouped by race and computed the minimum and maximum ages to identify the youngest and oldest drivers.

In [0]:
df_races = spark.read.csv(
    "/Volumes/gr5069/raw/f1_data/races.csv",
    header=True
)


In [0]:
from pyspark.sql.functions import col, to_date, floor, datediff, min, max

df_drivers2 = df_drivers.withColumn(
    "dob", to_date(col("dob"))
)

df_races2 = df_races.withColumn(
    "date", to_date(col("date"))
)

df_join = df_results.join(df_drivers2, "driverId") \
                    .join(df_races2, "raceId")

df_age = df_join.withColumn(
    "Age",
    floor(datediff(col("date"), col("dob")) / 365.25)
)

young_old = df_age.groupBy("raceId").agg(
    min("Age").alias("youngest_age"),
    max("Age").alias("oldest_age")
)

young_old.show()


Explanation: First, I converted the date of birth and race date columns to date format using to_date(). Then I joined the drivers, results, and races datasets using driverId and raceId. I created a new column called Age by computing the difference between race date and date of birth using datediff(), dividing by 365.25, and applying floor() to convert to years. Finally, I grouped by race and used min() and max() to find the youngest and oldest driver in each race.

Question 5: At any given race, how many podiums does each driver have? create three new columns to show - on any given race - the number of wins, the number of 2nd places, and the number of 3rd places for each driver

Logic: To calculate podium counts for each driver in each race, I used the results dataset and focused on the position column. Since some values in position are stored as non numeric strings such as \N, I first converted valid positions to integers and let invalid values become null. Then I grouped the data by raceId and driverId and counted how many times each driver finished in 1st, 2nd, or 3rd place. These counts became the new columns wins, second_places, and third_places.

In [0]:
from pyspark.sql.functions import col, sum, when, expr

df_results_clean = df_results.withColumn(
    "position_int",
    expr("try_cast(position as int)")
)

podiums = df_results_clean.groupBy(
    "raceId",
    "driverId"
).agg(
    sum(when(col("position_int") == 1, 1).otherwise(0)).alias("wins"),
    sum(when(col("position_int") == 2, 1).otherwise(0)).alias("second_places"),
    sum(when(col("position_int") == 3, 1).otherwise(0)).alias("third_places")
)

podiums.orderBy("raceId", "driverId").show()


Explanation: First, I created a new column called position_int using expr("try_cast(position as int)"). This converts valid finishing positions into integers and changes invalid values like \N into null instead of causing an error. Then I grouped the dataset by raceId and driverId so each row represents one driver in one race. Inside agg(), I used when() to check whether the driver finished in 1st, 2nd, or 3rd place, and sum() to count those finishes. This produced three new columns showing the number of wins, second places, and third places for each driver at each race.

Question 6:Continue exploring the data by answering your own question.

My question : Which driver had the fastest average lap time in each race?

Logic: To determine the fastest driver in each race, I used the lap_times dataset and calculated the average lap time for each driver within each race. Since lap time is measured in milliseconds, I grouped the data by raceId and driverId and computed the average lap time. Then I ranked drivers within each race and selected the minimum average lap time to identify the fastest driver.

In [0]:
from pyspark.sql.functions import col, avg, min

df_laps_clean = df_laptimes.withColumn(
    "milliseconds",
    col("milliseconds").cast("int")
)

avg_laps = df_laps_clean.groupBy(
    "raceId",
    "driverId"
).agg(
    avg("milliseconds").alias("avg_lap_time")
)

fastest = avg_laps.groupBy("raceId").agg(
    min("avg_lap_time").alias("fastest_avg_lap")
)

fastest.show()


Explanation: First, I converted the milliseconds column to integer to allow numeric calculations. Then I grouped the lap times by raceId and driverId and computed the average lap time using avg(). After that, I grouped again by raceId and used min() to find the smallest average lap time in each race. The smallest average lap time represents the fastest driver for that race.

In [0]:
fastest.write.csv('/Volumes/gr5069/sy3350/inclass/hw2')